In [ ]:
from datasets import Dataset

def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    stories = []
    story = ""
    for line in lines:
        line = line.strip()
        if line.isupper():  # Title lines are uppercase, so we skip them
            if story:
                stories.append(story.strip())
                story = ""
            continue
        story += " " + line if story else line
    
    if story:
        stories.append(story.strip())
    
    return Dataset.from_dict({"text": stories})

dataset = load_data("Aesops_fables.txt")

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2-finetuned')
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have a padding token, use EOS instead

In [ ]:
from transformers import DataCollatorForLanguageModeling, DataCollatorWithPadding

def tokenize_function(element):
    return tokenizer(element["text"])

# Tokenize dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

# Create a data collator that dynamically pads batches
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # or True if using masked language modeling
)

# Wrap it with DataCollatorWithPadding to enable adaptive padding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    collate_fn=data_collator
)

In [ ]:
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer

# Load model
model = GPT2LMHeadModel.from_pretrained('gpt2-finetuned')

training_args = TrainingArguments(
    output_dir='./checkpoint',  # Directory to store results
    num_train_epochs=3,             # Set to the number of epochs you want
    learning_rate=5e-4,
    per_device_train_batch_size=16, # Batch size per device
    gradient_accumulation_steps=4,  # Gradients are accumulated across 4 steps
    warmup_steps=500,               # Warm-up steps to avoid early training instability
    weight_decay=0.01,              # Regularization parameter
    logging_first_step=True,        # Log the first step
    
    logging_dir="./checkpoint/logs",
    logging_steps=100,              # Log every 100 steps
    logging_strategy="steps",
    
    report_to="tensorboard",
    
    save_steps=500,                 # Save checkpoint every 500 steps
    save_total_limit=2,             # Limit the number of saved checkpoints

    seed=42,                        # Random seed for reproducibility
    fp16=True,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Train the model
trainer.train()

# Save final model and tokenizer
model.save_pretrained("./checkpoint")
tokenizer.save_pretrained("./checkpoint")

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def generate_text(prompt, max_length=100, num_return_sequences=1, temperature=0.7, top_k=50, top_p=0.9):
    # Load the trained model and tokenizer
    model = GPT2LMHeadModel.from_pretrained("./checkpoint")
    tokenizer = GPT2Tokenizer.from_pretrained("./checkpoint")
    
    model.eval()
    
    # Tokenize input prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    
    # Generate text
    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True  # Enable sampling for more diverse outputs
    )
    
    # Decode and return generated text
    return [tokenizer.decode(seq, skip_special_tokens=True) for seq in output]

# Example usage
prompt = "Once upon a time in a small village,"
generated_texts = generate_text(prompt)
for i, text in enumerate(generated_texts):
    print(f"Generated Text {i+1}:\n{text}\n")